# SOLUCIÓN PROBLEMA 1 - PARCIAL Probabilidad y Estadística - iMAT-ICAI - 15 de diciembre 2021

# Vale el 30% de la nota del examen

## NO CAMBIES EL NOMBRE AL NOTEBOOK, SIMPLEMENTE ESCRIBE AQUÍ TUS DATOS
### Nombre: XXXXXXXXXXXXX
### Apellidos: XXXXXXXXXXXX XXXXXXXXXXX

En esta ocasión queremos estudiar la demanda diaria de energía eléctrica en la isla de la Palma. Hemos descargado de REE los datos de demanda desde enero de 2014 hasta principios de diciembre de 2021. 

A partir de esta información en bruto hemos preparado un dataset que contiene información de otras variables relacionadas con la demanda, como la temperatura (obtenida de AEMET) o información del calendario básica. Además, en el periodo de datos considerado hay dos sucesos muy importantes que también se han incluido en el dataset: el periodo COVID (inicio a medidados de marzo 2020) y la erupción del volcán de Cumbre Vieja (septiembre 2021).

A partir de la fecha (una serie de tipo time), hemos incluido en el fichero un conjunto de variables derivadas para facilitar el análisis (YEAR, MONTH  WEEKDAY y WEEKOFYEAR). La variable MONTH y YEAR son numéricas. La variable WEEKDAY también es numérica, con valores entre 1 (lunes) y 7 (domingo). WEEKOFYEAR indica la semana del año (variable numérica entre 1 y 53). Además, tenemos la variable COVID que vale 1 en el periodo donde se considera que la demanda estuvo fuertemente afectada por las medidas adoptadas para frenar la pandemia y las posteriores consecuencias económicas, y 0 en el resto del periodo. Algo similar tenemos para la variable VOLCAN, que vale 1 a partir de la fecha de inicio de la erupción.

Al principio de cada apartado te encontrarás una celda con el código que obtiene las variables a utilizar para responder ese apartado, generadas a partir del dataframe df_orig que se carga al principio con toda la información. También puede ser un nuevo dataframe df simplificado para facilitar los cálculos. Tendrás que trabajar con esas variables o ese nuevo dataframe, no con el dataframe df_orig directamente, para simplificar los cálculos.

Responde las preguntas utilizando instrucciones print() para mostrar claramente tu respuesta, explicando cómo lo resuelves. Utiliza comentarios en tu código para facilitar su lectura.

## Importamos todos los paquetes aquí por comodidad

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats

## Cargamos los datos con los que se trabaja

In [ ]:
df_orig = pd.read_csv('demanda_electricidad_LaPalma.csv', delimiter=',')

# parseamos la fecha (cadena) para que sea un datetime con formato dd/mm/yyyy
df_orig['DATE'] = pd.to_datetime(df_orig.DATE, format='%d/%m/%Y')

print('Tamaño de df_orig con los datos cargados:', df_orig.shape)
print('Dataframe con todos los datos leidos (filas: días, columnas: variables):')
print(df_orig)

### a) Representa la evolución temporal de la demanda diaria de la Palma, distinguiendo los datos de la zona del confinamiento por inicio de la pandemia de COVID, y el periodo desde el inicio de la erupción del volcán. Comenta lo que observas en la evolución temporal de la demanda. (1 punto)

In [ ]:
# variables con las que hay que trabajar
dem = df_orig.DEM.tolist()
dem_COVID = df_orig.DEM[df_orig.COVID == 1].tolist()
dem_VOLCAN = df_orig.DEM[df_orig.VOLCAN == 1].tolist()

# auxiliares útiles para la representación
fechas = df_orig.DATE.tolist()
fechas_COVID = df_orig.DATE[df_orig.COVID == 1].tolist()
fechas_VOLCAN = df_orig.DATE[df_orig.VOLCAN == 1].tolist()

In [ ]:
# respuesta

fig = plt.figure(1, figsize=(15,4)) # permite indicar el nº de la figura y las dimensiones (ancho y alto)
plt.plot(fechas, dem, label='Demanda')
plt.plot(fechas_COVID, dem_COVID, label='demanda zona COVID')
plt.plot(fechas_VOLCAN, dem_VOLCAN, label='demanda zona VOLCAN')
plt.legend()
plt.show()

print('La demanda ha estado subiendo durante los primeros años, en 2018 parece que se estabiliza, hasta la llegada de la pandemia.')
print('Es muy llamativo el desplome de la demanda en 2020 con el inicio de la pandemia COVID.')
print('También se observa un nuevo desplome, menos acusado con el inicio de la erupción del volcán.')

### b) Representa las distribuciones de la demanda diaria para todo el periodo, la demanda en la zona COVID y en la zona de erupción del VOLCAN. ¿Qué distribución es más asimétrica, la demanda COVID o la del VOLCAN?. Justifica las respuestas (1.5 puntos)

In [ ]:
# variables con las que hay que trabajar
dem = df_orig.DEM.tolist()
dem_COVID = df_orig.DEM[df_orig.COVID == 1].tolist()
dem_VOLCAN = df_orig.DEM[df_orig.VOLCAN == 1].tolist()

In [ ]:
# respuesta

nbins = 40 # número de intervalos, ajustar para buena visualización

fig = plt.figure(1, figsize=(12, 6)) # permite indicar el nº de la figura y las dimensiones (ancho y alto)
ax1 = plt.subplot(3,2,1)
sns.histplot(data=dem, bins=nbins, label='dem')
plt.ylabel('Nº días')
plt.legend()
ax2 = plt.subplot(3,2,3, sharex = ax1)
sns.histplot(data=dem_COVID, color='orange', bins=nbins, label='dem_COVID')
plt.legend()
plt.ylabel('Nº días')
ax3 = plt.subplot(3,2,5, sharex = ax1)
sns.histplot(data=dem_VOLCAN, color='green', bins=nbins-20, label='dem_VOLCAN')
plt.legend()
plt.ylabel('Nº días')

ax4 = plt.subplot(3,2,2)
sns.boxplot(data=dem, orient='horizontal',  notch=True)
ax5 = plt.subplot(3,2,4, sharex = ax1)
sns.boxplot(data=dem_COVID, orient='horizontal', color='orange')
ax6 = plt.subplot(3,2,6, sharex = ax1)
s = sns.boxplot(data=dem_VOLCAN, orient='horizontal', color='green')

plt.show()

print('La distribución de la demanda diaria en la zona COVID es más asimétrica que la demanda de la zona volcán,'
     ' se observa claramente tanto en los histogramas como en los diagramas de cajas. La asimetría es positiva.'
     ' Es posible confirmarlo calculando el coeficiente de asimetría, pero no parece necesario.')

### c) Calcula la demanda diaria media en abril de 2019 y compárala con la media de abril de 2020. A partir de esos dos valores, ¿en cuánto se puede cuantificar el impacto del COVID en abril de 2020? (1.5 puntos)

In [ ]:
# variables con las que hay que trabajar
dem_abril_2019 = df_orig.DEM[(df_orig.YEAR==2019) & (df_orig.MONTH==4)]
dem_abril_2020 = df_orig.DEM[(df_orig.YEAR==2020) & (df_orig.MONTH==4)]

In [ ]:
# respuesta

#medias
med_2019 = np.mean(dem_abril_2019)
med_2020 = np.mean(dem_abril_2020)
print('la demanda media diaria de abril 2019 fue',med_2019, ', superior a la demanda media de abril 2020, que fue', med_2020)

impacto = 100 * (med_2019 - med_2020) / med_2019
print('A partir de las medias, el impacto se puede cuantificar en un descenso medio diario de un', impacto, '%')

# visualmente
fig = plt.figure(1, figsize=(15,4))
plt.plot(dem_abril_2019, label='dem abril 2019')
plt.plot(dem_abril_2020, label='dem abril 2020')
plt.grid()
plt.legend()
plt.show()

### d) El día de la semana suele influir en el consumo de electricidad. Con los datos entre 2017 y 2020, analiza la distribución de la demanda según los días de la semana. ¿En qué día de la semana la demanda diaria suele ser menor? ¿Dentro del periodo considerado, en qué dia de la semana se produjo el día de mayor consumo diario? (2 puntos)

In [ ]:
# variables con las que hay que trabajar
df_entre_2017_y_2021 = df_orig[(df_orig.YEAR > 2016) & (df_orig.YEAR < 2020)]
df_sel = df_entre_2017_y_2021.loc[:,['WEEKYEAR', 'WEEKDAY', 'DEM']]
df_dem_diasem = df_sel.pivot(index='WEEKYEAR', columns='WEEKDAY')

#---------------------------------------------------------------------------
# necesitas trabajar con este dataframe df, que tiene en cada columna la demanda en un día de la semana diferente <---------------
#---------------------------------------------------------------------------
df = df_dem_diasem['DEM']
df.columns = ['LUNES', 'MARTES','MIERCOLES','JUEVES','VIERNES','SABADO','DOMINGO']
print(df)

In [ ]:
# respuesta

# pintamos un boxplot por diasem para verlo
df.boxplot(figsize=(10,6)) 
plt.title('Distribución de la demanda según dias de la semana')
plt.show()

df.hist(figsize=(10,8), bins = 20, sharex= True)
plt.show()

print('En los domingos la demanda suele ser claramente menor.')
print('También se observa una demanda inferior los sábados, comparados con el resto de laborables, pero no tan baja como en domingo.')
print('El día de mayor consumo fue un jueves, como se puede observar claramente con el atípico superíor.')

### La demanda depende de la temperatura, pero de una forma caprichosa, debido al uso que hacemos de la calefacción y el aire acondicionado para mantener la temperatura en nuestra zona de confort. Analiza la relación existente entre la demanda y la temperatura con los datos entre 2017 y 2020. Distingue entre tres tramos posibles de temperaturas: por debajo de 20 grados, entre 20 y 21 grados y mayor o igual que 21 grados. 

In [ ]:
# variables con las que hay que trabajar
df_antes_2020 = df_orig[(df_orig.YEAR > 2016) & (df_orig.YEAR < 2020)]

# tramos a considerar
dem_inf_20  = df_antes_2020.DEM[(df_antes_2020.TEMP < 20)].tolist()
temp_inf_20 = df_antes_2020.TEMP[(df_antes_2020.TEMP < 20)].tolist()

dem_20_22  = df_antes_2020.DEM[(df_antes_2020.TEMP >= 20)  & (df_antes_2020.TEMP < 21)].tolist()
temp_20_22 = df_antes_2020.TEMP[(df_antes_2020.TEMP >= 20) & (df_antes_2020.TEMP < 21)].tolist()

dem_sup_22  = df_antes_2020.DEM[(df_antes_2020.TEMP >= 21)].tolist()
temp_sup_22 = df_antes_2020.TEMP[(df_antes_2020.TEMP >= 21)].tolist()

### e) Representa visualmente, en una misma gráfica, la relación entre la demanda y la temperatura en los tres tramos indicados mediante una nube de puntos ¿Las demandas más elevadas aparecen cuando hace mucho calor o mucho frío?. Si, estando la temperatura en el último tramo, ésta sube un grado, ¿qué se debería esperar que pase en la demanda (se mantendrá igual, subirá o bajará)? ¿Y en el resto de tramos? (2 puntos)

In [ ]:
# respuesta

# scatter para cada tramo en el mismo gráfico
fig = plt.figure(1, figsize=(10,4))
plt.plot(temp_inf_20, dem_inf_20, 'b.')
plt.plot(temp_20_22, dem_20_22, 'k.')
plt.plot(temp_sup_22, dem_sup_22, 'r.')
plt.grid()
plt.xlabel('TEMP (ºC)')
plt.ylabel('DEM')
plt.show()

print('\nCuando hace mucho calor aparecen los valores más altos de demanda (por encima de 850)')

print(' Si la temperatura sube en el tramo rojo, la demanda normalmente subirá, seguramente por el uso de aire acondicionado')
print(' Si la temperatura sube en el tramo negro, la demanda no parece que cambie de forma sistemática')
print(' Si la temperatura sube en el tramo azul, la demanda normalmente bajará, seguramente al dejar de usarse calefacción eléctrica')


### f) Calcula el coeficiente de correlación lineal entre la temperatura y la demanda en cada uno de los tres tramos. Relaciona los tres valores calculados con lo observado en el apartado anterior. (2 puntos)

In [ ]:
# respuesta

# coeffs correlación lineal en cada tramo
corr_inf_20 = np.corrcoef(temp_inf_20, dem_inf_20)
corr_20_22 = np.corrcoef(temp_20_22, dem_20_22)
corr_sup_22 = np.corrcoef(temp_sup_22, dem_sup_22)
   
print('Correlaciones:')
print('  - corr (zona azul)  =', corr_inf_20[0,1])
print('  - corr (zona negra) =', corr_20_22[0,1])
print('  - corr (zona roja)  =', corr_sup_22[0,1])

print('\nTiene sentido según lo observado en las nubes de puntos. Hay una zona intermedia (entre 20 y 21 grados) en la que no existe relación entre la temperatura y la demanda'
     ', confirmado por un coef. de correlación prácticamente nulo. En la zona azul la correlación es negativa, es decir, existe una relación inversa entre ambas variables. '
     'Por último, en la zona roja la relación es directa (coef. correlación positivo) y mucho más fuerte que en la zona de temperaturas bajas.')
